# UCAP -- SAM 3 + UniDepth v2 (monocular metric depth)

**End-to-end depth-gated anonymizer for a single egocentric clip** -- the monocular variant of the Utility Carve-out of Anonymisation Protocol (UCAP) evaluated in the accompanying paper.

1. **SAM 3** segments the privacy concepts (`face`, `license plate`) on every frame -- recall-first, score threshold 0.6.
2. **UniDepth v2** predicts metric depth with no calibration -> the *near-field* mask (< 1.1 m), the interaction zone to **preserve**.
3. The output redacts **privacy MINUS near**; every frame where the carve-out is significant is written to a **review log** + conflict video (the **M2 conflict / human-review** metric). Each flagged frame then takes a 3-way human decision: drop the blur / keep the blur / mark the frame unusable.

> Runtime: Google Colab, single GPU (L4 recommended, T4 works -- two models are resident).
> The maximum-utility output video is **not** the shareable deliverable until its flagged frames pass review.

## Step 1 -- runtime check

In [ ]:
# Runtime check -- confirm a GPU is attached (T4 is fine for stereo; the
# DAC/UniDepth notebook is happier on an L4 because two models are resident).
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU:", name, "| total VRAM (GB):", round(total, 2))
    if not any(g in name for g in ("T4", "L4", "A100", "L40")):
        print("NOTE: unrecognised GPU -- numbers will not match the T4 headline.")
else:
    print("No GPU. Runtime > Change runtime type > GPU (T4 or L4).")

## Step 2 -- install

In [ ]:
# Installs. SAM 3 (ultralytics) + UniDepth (the in-loop wide-FoV metric
# depth backend). Pillow is NOT upgraded (>=12 breaks Colab imports).
!pip install -q -U ultralytics huggingface_hub
!pip install -q -U opencv-python-headless imageio imageio-ffmpeg matplotlib
# UniDepth v2 -- pip from source; predicts its own intrinsics (no calibration).
# If this line errors, the depth cell prints a clear fallback note.
!pip install -q "git+https://github.com/lpiccinelli-eth/UniDepth.git"
print("install done")

## Step 3 -- Hugging Face auth (gated SAM 3)

In [ ]:
# Hugging Face auth -- facebook/sam3 is a GATED repo.
#   1. Request access once: https://huggingface.co/facebook/sam3
#   2. Token (read scope): https://huggingface.co/settings/tokens
#   3. Colab: Secrets panel (key icon) -> add secret HF_TOKEN -> enable for notebook.
# NEVER hardcode a token in a cell -- it leaks the moment the file is shared.
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
    print("HF auth OK -- gated SAM 3 weights accessible.")
except Exception as e:
    print("HF auth NOT set up:", repr(e))
    print("SAM 3 (sam3.pt) will fail to download without it.")

## Step 4 -- config
Everything tunable lives here -- prompts, the near threshold, the carve policy, the review threshold.

In [ ]:
# ============================ CONFIG (edit me) =============================
# Privacy concepts SAM 3 will segment and blur. The released evaluation used the
# EgoBlur-matched pair below; widen by adding noun-phrase prompts if needed.
PROMPTS = ["face", "license plate"]
# PROMPTS += ["document", "computer screen", "phone screen", "tattoo"]

SCORE_THRESHOLD = 0.6        # SAM 3 detection threshold (lower = higher recall)
MAX_FRAMES      = None        # cap for a quick pass; set None for the whole clip

# --- Near zone (the interaction zone to PRESERVE) -------------------------
# UniDepth returns METRIC depth in metres, so the threshold is literal:
NEAR_METERS     = 1.1        # "arm's reach"; pixels closer than this are preserved
USE_METRIC      = True       # True: depth < NEAR_METERS. False: nearest NEAR_PERCENT%.
NEAR_PERCENT    = 40         # relative fallback if USE_METRIC=False

# --- Carve policy + review threshold (this is M2) -------------------------
PRESERVE_POLICY  = "carve"   # "carve" = blur MINUS near (requested) | "failsafe" = blur all, still flag
OVERLAP_THRESHOLD = 0.2     # a privacy instance >=15% inside near -> flagged (matches pipeline.py)

# --- Redaction style ------------------------------------------------------
REDACTION       = "fill"     # "blur" | "pixelate" | "fill"
BLUR_KSIZE      = 41         # odd; Gaussian kernel for "blur"
PIXELATE_BLOCKS = 16         # for "pixelate"
FILL_BGR        = (255, 0, 0)  # solid colour for "fill" (OpenCV BGR)

# --- Outputs --------------------------------------------------------------
OUT_VIDEO       = "/content/anon_dac.mp4"
REVIEW_LOG      = "/content/review_log_dac.json"
SAMPLE_FIG_PATH = "/content/anon_dac_sample.jpg"
DEPTH_BACKEND   = "unidepth"   # "unidepth" (runs here) | "dac" (see Step 7b)
LENS            = "unknown"     # label this clip for per-lens analysis: "wide-angle" | "fisheye" | "unknown"
print("config set | policy:", PRESERVE_POLICY, "| near:",
      (str(NEAR_METERS) + " m") if USE_METRIC else ("nearest " + str(NEAR_PERCENT) + "%"))

## Step 5 -- source clip
A single fisheye / wide-FoV egocentric video.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Source clip -- a single (fisheye / wide-FoV) egocentric video.
import cv2, numpy as np
VIDEO_PATH = '/content/drive/MyDrive/Licencjat/stock/513000_Gopro Waves Surfboard Pov_By_Ross_Hillier_Artlist_HD.mp4'
VIDEO_PATH3 = '/content/drive/MyDrive/Licencjat/ego_data/stereo_right.mp4'
VIDEO_PATH2 = '/content/drive/MyDrive/Licencjat/ego_data/fisheye_cam2_small.mp4'
# Option 1 -- Drive:
# from google.colab import drive; drive.mount("/content/drive")
# VIDEO_PATH = "/content/drive/MyDrive/fisheye_cam2_small.mp4"
# Option 2 -- upload:
# from google.colab import files; up = files.upload(); VIDEO_PATH = list(up.keys())[0]
# Option 3 -- the repo clip (upload clips/ropedia/fisheye_cam2_small.mp4 first):
# VIDEO_PATH = "fisheye_cam2_small.mp4"

# Option 4 -- synthetic fallback so the harness runs without data (coverage ~0):
if VIDEO_PATH is None:
    VIDEO_PATH = "synthetic_clip.mp4"
    vw = cv2.VideoWriter(VIDEO_PATH, cv2.VideoWriter_fourcc(*"mp4v"), 30, (640, 480))
    for i in range(60):
        a = np.full((480, 640, 3), 230, np.uint8)
        x = 20 + i * 8
        cv2.rectangle(a, (x, 200), (x + 110, 320), (40, 90, 200), -1)
        vw.write(a)
    vw.release()
    print("Synthetic clip -- use Option 1/2/3 for real numbers.")
print("VIDEO_PATH:", VIDEO_PATH)

## Step 6 -- helpers

In [ ]:
# Shared helpers: VRAM, redaction, and the figure utility.
import time, gc, json
import numpy as np
import cv2
import matplotlib.pyplot as plt

def reset_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def peak_vram_gb():
    return round(torch.cuda.max_memory_allocated() / 1e9, 3) if torch.cuda.is_available() else None

def apply_redaction(frame_bgr, mask):
    """Redact the True pixels of `mask` using the REDACTION mode from config.
    Mask-aware: only masked pixels change, so background detail is untouched."""
    out = frame_bgr.copy()
    if mask is None or not mask.any():
        return out
    if REDACTION == "fill":
        out[mask] = FILL_BGR
    elif REDACTION == "pixelate":
        h, w = frame_bgr.shape[:2]
        s = max(1, PIXELATE_BLOCKS)
        small = cv2.resize(frame_bgr, (max(1, w // s), max(1, h // s)),
                           interpolation=cv2.INTER_LINEAR)
        pix = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
        out[mask] = pix[mask]
    else:  # "blur" (default) -- matches code/pipeline.py blur_regions
        k = int(BLUR_KSIZE) | 1
        out[mask] = cv2.GaussianBlur(frame_bgr, (k, k), 0)[mask]
    return out

def depth_to_color(depth):
    """Normalised inferno heatmap of a depth/disparity map for figures."""
    d = np.asarray(depth, dtype="float32")
    finite = np.isfinite(d) & (d > 0)
    if not finite.any():
        return np.zeros((*d.shape, 3), "uint8")
    lo, hi = np.percentile(d[finite], [2, 98])
    n = np.clip((d - lo) / max(hi - lo, 1e-6), 0, 1)
    return (plt.cm.inferno(n)[:, :, :3] * 255).astype("uint8")

SAMPLES = []    # (idx, redacted_bgr, depth, near_mask, privacy_mask) for the figure
CONFLICTS = []  # (idx, time_s, redacted_bgr, deblurred_mask) for the conflict reel (Step 10b)
FRAMELOG = []    # per-frame PII-safe aggregates for the rich result log
print("helpers ready")

## Step 7 -- depth backend (UniDepth)
Metric depth, no calibration. On a T4 swap `UNIDEPTH_ID` to `...vitb14` or `...vits14` if VRAM is tight.

In [ ]:
# === Depth backend: UniDepth v2 (monocular, metric, predicts intrinsics) ====
# This is the in-loop wide-FoV depth model. It needs NO calibration -- it infers
# the camera model itself, which is why it works on an arbitrary fisheye clip
# where DAC would need the intrinsics (see Step 7b for DAC).
import torch, numpy as np, cv2

UNIDEPTH_ID = "lpiccinelli/unidepth-v2-vitl14"   # vits14 / vitb14 are lighter (T4)
_udep = None
def _load_unidepth():
    global _udep
    if _udep is None:
        from unidepth.models import UniDepthV2
        _udep = UniDepthV2.from_pretrained(UNIDEPTH_ID).to(
            "cuda" if torch.cuda.is_available() else "cpu").eval()
        print("UniDepth loaded:", UNIDEPTH_ID)
    return _udep

def estimate_depth(frame_bgr, idx):
    """Return a metric depth map (H, W) in METRES for one BGR frame."""
    model = _load_unidepth()
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    t = torch.from_numpy(rgb).permute(2, 0, 1)        # C,H,W uint8
    with torch.no_grad():
        pred = model.infer(t)
    return pred["depth"].squeeze().cpu().numpy().astype("float32")

def near_mask(depth):
    """Bool mask of the near interaction zone to PRESERVE (smaller depth = closer)."""
    d = np.asarray(depth, dtype="float32")
    if USE_METRIC:
        return (d > 0) & (d < NEAR_METERS)
    valid = d[np.isfinite(d) & (d > 0)]
    if valid.size == 0:
        return np.zeros(d.shape, bool)
    return (d > 0) & (d <= np.percentile(valid, NEAR_PERCENT))

# Smoke test on frame 0:
_cap = cv2.VideoCapture(VIDEO_PATH); _ok, _f = _cap.read(); _cap.release()
if _ok:
    _d = estimate_depth(_f, 0)
    print("depth map:", _d.shape, "| metres range %.2f..%.2f" % (float(np.nanmin(_d)), float(np.nanmax(_d))),
          "| near pixels:", int(near_mask(_d).sum()))

## Step 7b -- DAC (cite / ceiling, separate runtime)

In [ ]:
# === (Optional, Step 7b) DAC -- Depth Any Camera, the fisheye-native model ===
# DAC is the academically-correct large-FoV METRIC model and the right thing to
# CITE for fisheye depth. It is NOT wired into the loop above because:
#   * it pins python 3.9 / torch 1.13.1+cu116 with custom CUDA ops, which fights
#     this torch-2.x runtime (don't downgrade -- it would break SAM 3 + UniDepth);
#   * its demo is dataset-coupled: it needs the camera's intrinsics (fx, cx, cy)
#     and an ERP / fisheye-grid lookup -- the calibration you don't have for the
#     ropedia sample.
# Use this in a SEPARATE, clean Colab runtime to get DAC's own depth on its OWN
# sample (intrinsics known), for a citation + a "ceiling" figure.
#
#   !git clone https://github.com/yuliangguo/depth_any_camera.git
#   %cd depth_any_camera
#   # follow the repo README: conda env (py3.9, torch 1.13.1+cu116),
#   #   pip install -r requirements.txt ; cd dac/models/ops ; pip install -e .
#   # download a checkpoint pair into checkpoints/ (see the repo's HF links), then:
#   !python demo/demo_dac_single.py \
#       --config-file checkpoints/dac_swinl_indoor.json \
#       --model-file  checkpoints/dac_swinl_indoor.pt \
#       --sample-file demo/input/scannetpp_sample.json \
#       --out-dir     demo/output
#
# To put DAC IN THE LOOP here later: implement estimate_depth(frame, idx) to call
# DAC with your camera's cam_params {fx, cx, cy} + ERP grid (mirror the sample
# JSON in demo/input/), then set DEPTH_BACKEND="dac". The rest of the pipeline is
# unchanged -- it only consumes a metres-valued depth map.
print("DAC is a separate-runtime / cite step. In-loop backend stays:", DEPTH_BACKEND)

## Step 8 -- load SAM 3

In [ ]:
# Load SAM 3 once -- Ultralytics video-native predictor (axis B1, the
# BrainHack-validated path). `sam3.pt` is the gated facebook/sam3 weight.
from huggingface_hub import hf_hub_download
from ultralytics.models.sam import SAM3VideoSemanticPredictor

SAM3_PT = hf_hub_download(repo_id="facebook/sam3", filename="sam3.pt", local_dir=".")
print("sam3.pt:", SAM3_PT)

# half=True -> FP16 (the T4 path); retina_masks=True -> full-res instance masks
# aligned to orig_img; score_threshold_detection is the recall lever (0.25-0.6).
sam3_overrides = dict(task="segment", mode="predict", model="sam3.pt",
                      half=True, save=False, retina_masks=True, verbose=False)
sam3 = SAM3VideoSemanticPredictor(overrides=sam3_overrides,
                                  score_threshold_detection=SCORE_THRESHOLD)
print("SAM 3 ready. Prompts:", PROMPTS)

## Step 9 -- run the pipeline
One SAM 3 streaming pass; depth + carve + review log per frame.

In [ ]:
# === The pipeline: SAM 3 masks -> depth near-mask -> (blur MINUS near) ======
# One streaming SAM 3 pass over the LEFT/only video. Per frame:
#   privacy = union of SAM 3 instance masks for PROMPTS
#   near    = near_mask(estimate_depth(frame, idx))    # the zone to preserve
#   blur    = privacy & ~near   (PRESERVE_POLICY="carve", the requested behaviour)
#          or privacy           (PRESERVE_POLICY="failsafe": redact all, still flag)
# A frame is flagged for review when ANY privacy instance has >= OVERLAP_THRESHOLD
# of its area inside the near zone -- exactly resolve_conflicts() in pipeline.py.
def _instances(r, H, W):
    out = []
    m = getattr(r, "masks", None)
    if m is not None and getattr(m, "data", None) is not None and len(m.data) > 0:
        arr = m.data.to("cpu").numpy().astype(bool)            # (N, h, w)
        for k in range(arr.shape[0]):
            im = arr[k]
            if im.shape != (H, W):
                im = cv2.resize(im.astype("uint8"), (W, H),
                                interpolation=cv2.INTER_NEAREST).astype(bool)
            out.append(im)
    return out

def process_video(video_path, out_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    review = []
    SAMPLES.clear()
    CONFLICTS.clear()
    FRAMELOG.clear()
    n = n_blur = n_flag = 0
    sample_at = set(np.linspace(0, max(0, (MAX_FRAMES or 120) - 1), 3, dtype=int).tolist())
    reset_vram(); t0 = time.perf_counter()

    for r in sam3(source=video_path, text=PROMPTS, stream=True):
        frame = r.orig_img.copy()                              # BGR, full size
        Hf, Wf = frame.shape[:2]
        insts = _instances(r, Hf, Wf)
        privacy = np.zeros((Hf, Wf), bool)
        for im in insts:
            privacy |= im

        try:
            depth = estimate_depth(frame, n)
            near = near_mask(depth)
            if near.shape != (Hf, Wf):
                near = cv2.resize(near.astype("uint8"), (Wf, Hf),
                                  interpolation=cv2.INTER_NEAREST).astype(bool)
        except Exception as e:
            if n == 0:
                print("depth error (continuing with empty near-mask):", repr(e))
            depth, near = np.zeros((Hf, Wf), "float32"), np.zeros((Hf, Wf), bool)

        blur = (privacy & ~near) if PRESERVE_POLICY == "carve" else privacy.copy()

        # --- per-frame PII-safe aggregates for the rich log --------------
        _dv = np.isfinite(depth) & (depth > 0)
        FRAMELOG.append({
            "f": n, "t": round(n / fps, 2),
            "near_frac": round(float(near.mean()), 4),
            "priv_frac": round(float(privacy.mean()), 4),
            "blur_frac": round(float(blur.mean()), 4),
            "preserved_frac": round(float((privacy & near).mean()), 4),
            "depth_med_m": round(float(np.median(depth[_dv])), 3) if _dv.any() else 0.0,
            "depth_valid": round(float(_dv.mean()), 3),
            "n_inst": len(insts),
        })

        flagged, max_ov = False, 0.0
        for im in insts:
            d = int(im.sum())
            if not d:
                continue
            ov = float((im & near).sum()) / d
            max_ov = max(max_ov, ov)
            if ov >= OVERLAP_THRESHOLD:
                flagged = True
        if flagged:
            n_flag += 1
            review.append({"frame": n, "time_s": round(n / fps, 3),
                           "max_overlap": round(max_ov, 3),
                           "deblurred_px": int((privacy & near).sum()),
                           "privacy_px": int(privacy.sum())})

        out = apply_redaction(frame, blur)
        if blur.any():
            n_blur += 1
        writer.write(out)
        if n in sample_at:
            SAMPLES.append((n, out.copy(), depth, near, privacy))
        if flagged:
            CONFLICTS.append((n, round(n / fps, 3), out.copy(), (privacy & near)))
        n += 1
        if MAX_FRAMES and n >= MAX_FRAMES:
            break
    writer.release()
    dt = time.perf_counter() - t0
    def _agg(key):
        vals = [d[key] for d in FRAMELOG]
        if not vals:
            return {}
        arr = np.array(vals, dtype="float64")
        return {"mean": round(float(arr.mean()), 4), "median": round(float(np.median(arr)), 4),
                "min": round(float(arr.min()), 4), "max": round(float(arr.max()), 4),
                "std": round(float(arr.std()), 4)}
    aggregates = {k: _agg(k) for k in
                  ("near_frac", "priv_frac", "blur_frac", "preserved_frac",
                   "depth_med_m", "depth_valid", "n_inst")}
    frames_with_det = sum(1 for d in FRAMELOG if d["n_inst"] > 0)
    stats = {"frames": n, "fps_pipeline": round(n / dt, 2) if dt else 0.0,
             "width": W, "height": H, "src_fps": round(float(fps), 3),
             "frames_with_blur": n_blur, "frames_with_detection": frames_with_det,
             "flagged_frames": n_flag,
             "m2_conflict_rate": round(n_flag / n, 4) if n else 0.0,
             "peak_vram_gb": peak_vram_gb(), "out": out_path,
             "aggregates": aggregates}
    return review, stats

REVIEW, STATS = process_video(VIDEO_PATH, OUT_VIDEO)
print(json.dumps(STATS, indent=2, default=str))

## Step 10 -- review log + M2

In [ ]:
# === Rich review log + the M2 conflict/review number =======================
# One downloadable JSON per video: config + environment + per-frame aggregates +
# conflict events + M2. All PII-SAFE (numbers only, no pixels). Doubles as the
# raw data for the Results section.
import os, platform

def _env():
    info = {"python": platform.python_version(), "depth_backend": DEPTH_BACKEND}
    try:
        import torch as _t
        info["torch"] = _t.__version__
        if _t.cuda.is_available():
            info["gpu"] = _t.cuda.get_device_name(0)
            info["gpu_total_vram_gb"] = round(_t.cuda.get_device_properties(0).total_memory / 1e9, 2)
        else:
            info["gpu"] = "cpu"
    except Exception as e:
        info["torch_err"] = repr(e)[:120]
    try:
        import ultralytics as _u
        info["ultralytics"] = _u.__version__
    except Exception:
        pass
    try:
        info["unidepth_id"] = UNIDEPTH_ID
    except Exception:
        pass
    return info

def _episodes(review):
    frames = sorted(e["frame"] for e in review)
    eps, run = [], []
    for f in frames:
        if run and f == run[-1] + 1:
            run.append(f)
        else:
            if run:
                eps.append(run)
            run = [f]
    if run:
        eps.append(run)
    return eps

_eps = _episodes(REVIEW)
_ovs = [e["max_overlap"] for e in REVIEW]
event_summary = {
    "flagged_frames": STATS["flagged_frames"],
    "episodes": len(_eps),
    "longest_episode_frames": max((len(e) for e in _eps), default=0),
    "max_overlap_mean": round(float(np.mean(_ovs)), 3) if _ovs else 0.0,
    "max_overlap_max": round(float(np.max(_ovs)), 3) if _ovs else 0.0,
}

_series = FRAMELOG
if len(_series) > 600:
    step = (len(_series) + 599) // 600
    _series = _series[::step]

CONFIG = {"prompts": PROMPTS, "score_threshold": SCORE_THRESHOLD, "max_frames": MAX_FRAMES,
          "near_meters": NEAR_METERS, "use_metric": USE_METRIC, "near_percent": NEAR_PERCENT,
          "preserve_policy": PRESERVE_POLICY, "overlap_threshold": OVERLAP_THRESHOLD,
          "redaction": REDACTION}

log = {"video": VIDEO_PATH, "video_name": os.path.basename(VIDEO_PATH), "lens": LENS,
       "config": CONFIG, "environment": _env(), "stats": STATS,
       "event_summary": event_summary, "series_sampled": _series, "events": REVIEW}
with open(REVIEW_LOG, "w", encoding="utf-8") as fh:
    json.dump(log, fh, indent=2, default=str)

print("M2 conflict/review rate:", STATS["m2_conflict_rate"],
      "(" + str(STATS["flagged_frames"]) + "/" + str(STATS["frames"]) + " frames flagged)")
print("lens:", LENS, "| episodes:", event_summary["episodes"],
      "| longest:", event_summary["longest_episode_frames"], "frames")
print("depth_valid mean:", STATS["aggregates"].get("depth_valid", {}).get("mean"),
      "| near_frac mean:", STATS["aggregates"].get("near_frac", {}).get("mean"))
print("rich review log ->", REVIEW_LOG)
print()
print("first review events (frame @ time_s : max_overlap):")
for e in REVIEW[:12]:
    print(f"  frame {e['frame']:4d} @ {e['time_s']:7.2f}s : "
          f"overlap {e['max_overlap']:.2f}, deblurred {e['deblurred_px']} px")
if not REVIEW:
    print("  (none -- no privacy surface entered the near zone at this threshold)")


## Step 10b -- conflict review reel
A video of **only the conflict frames** (the ones in the review log), each with a
**red border around the de-blurred areas** -- the sensitive regions kept sharp
because they sit in the near interaction zone. This is the reviewer's visual
worklist. It reads `CONFLICTS`, which **Step 9 now stashes**, so **re-run Step 9
once** after adding this before running this cell.

In [ ]:
# === Conflict review reel: red border around every de-blurred area =========
# A video of ONLY the conflict frames (those in the review log), each showing the
# REDACTED output with a RED outline around the de-blurred regions -- the privacy
# pixels kept sharp because they fell in the near / interaction zone. Every red
# outline is something a human should eyeball. Fed by CONFLICTS, which Step 9 now
# stashes -> re-run Step 9 first if you have not since adding this cell.
REEL_VIDEO = OUT_VIDEO.replace(".mp4", "_conflicts.mp4")
REEL_FPS   = 4              # low fps so each conflict frame lingers (~1/REEL_FPS s)
BORDER_BGR = (0, 0, 255)    # red (OpenCV BGR)
BORDER_PX  = 3

if not CONFLICTS:
    print("No conflict frames to render -- review log was empty. "
          "(Run Step 9 first, or no privacy surface met the near zone.)")
else:
    H, W = CONFLICTS[0][2].shape[:2]
    rw = cv2.VideoWriter(REEL_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), REEL_FPS, (W, H))
    for idx, t_s, red, deblurred in CONFLICTS:
        canvas = red.copy()
        m = np.asarray(deblurred).astype("uint8")
        if m.shape[:2] != canvas.shape[:2]:
            m = cv2.resize(m, (canvas.shape[1], canvas.shape[0]),
                           interpolation=cv2.INTER_NEAREST)
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(canvas, cnts, -1, BORDER_BGR, BORDER_PX)
        label = "frame %d  t=%.2fs  de-blurred regions: %d" % (idx, t_s, len(cnts))
        cv2.rectangle(canvas, (0, 0), (W, 26), (0, 0, 0), -1)
        cv2.putText(canvas, label, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                    (255, 255, 255), 1, cv2.LINE_AA)
        rw.write(canvas)
    rw.release()
    print("conflict reel ->", REEL_VIDEO, "| frames:", len(CONFLICTS), "| fps:", REEL_FPS)
    print("Download:  from google.colab import files; files.download('" + REEL_VIDEO + "')")

## Step 11 -- sample figure (redacted only)

In [ ]:
# === Sample figure -- REDACTED output only (no raw PII; context/README.md) ==
# Columns: redacted frame | depth/disparity heatmap | near-mask (cyan) over the
# redacted frame. Safe to put in the paper / show the supervisor.
if SAMPLES:
    fig, axes = plt.subplots(len(SAMPLES), 3, figsize=(13, 4 * len(SAMPLES)))
    if len(SAMPLES) == 1:
        axes = axes[None, :]
    for row, (idx, red, depth, near, priv) in enumerate(SAMPLES):
        rgb = cv2.cvtColor(red, cv2.COLOR_BGR2RGB)
        axes[row, 0].imshow(rgb); axes[row, 0].set_title("frame %d -- redacted" % idx)
        axes[row, 1].imshow(depth_to_color(depth)); axes[row, 1].set_title("depth / disparity")
        ov = rgb.copy()
        if near is not None and near.any():
            ov[near] = (0.5 * ov[near] + 0.5 * np.array([0, 255, 255])).astype("uint8")
        axes[row, 2].imshow(ov); axes[row, 2].set_title("near zone (preserved) = cyan")
        for c in range(3):
            axes[row, c].axis("off")
    plt.tight_layout(); plt.savefig(SAMPLE_FIG_PATH, dpi=110, bbox_inches="tight")
    plt.show()
    print("figure ->", SAMPLE_FIG_PATH)
else:
    print("no samples captured")

## Step 12 -- save / download

In [ ]:
# === Save / download the three outputs ======================================
# Recommended: copy to Drive so they survive the runtime. Otherwise download.
print("outputs:")
print("  video :", OUT_VIDEO)
print("  log   :", REVIEW_LOG)
print("  figure:", SAMPLE_FIG_PATH)

# --- Option A: copy to Google Drive (uncomment) ---
# from google.colab import drive; drive.mount("/content/drive")
# import shutil, os
# dst = "/content/drive/MyDrive/anon_out"; os.makedirs(dst, exist_ok=True)
# for p in (OUT_VIDEO, REVIEW_LOG, SAMPLE_FIG_PATH): shutil.copy(p, dst)
# print("copied to", dst)

# --- Option B: download to your machine (uncomment) ---
# from google.colab import files
# files.download(OUT_VIDEO); files.download(REVIEW_LOG); files.download(SAMPLE_FIG_PATH)

---
### After it runs
1. Watch `OUT_VIDEO`: privacy surfaces blurred, the near interaction zone left sharp.
2. Open `REVIEW_LOG`: each event is a frame a human should check -- this is **M2** (conflict / human-review). The printed `m2_conflict_rate` is the headline number; pair it with M1/M3 (leak) for the trade-off.
3. Tune from the **config cell**: `PROMPTS`, the near threshold, `OVERLAP_THRESHOLD`, and `PRESERVE_POLICY` ("carve" vs "failsafe").

**The privacy nuance (your thesis spine).** `"carve"` keeps a near, held object sharp -- but if that object IS a privacy surface (a phone screen showing a face in your hand), carving it un-blurs PII -> an M1/M3 leak. `"failsafe"` blurs all PII and only *flags* the overlap. The review log is what lets you defend either choice with a number.